In [32]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup

URL = "https://api.divar.ir/v8/postlist/w/search"


headers = {
    "Content-Type": "application/json"
}

payload = {
    "city_ids": ["1"],
    "source_view": "FILTER",
    "pagination_data": {
        "@type": "type.googleapis.com/post_list.PaginationData",
        "page": 1,
        "layer_page": 1
    },
    "disable_recommendation": False,
    "map_state": {
        "camera_info": {
            "bbox": {}
        }
    },
    "search_data": {
        "form_data": {
            "data": {
                "brand_model": {
                    "repeated_string": {
                        "value": [
                            "Peugeot 206 5"
                        ]
                    }
                },
                "category": {
                    "str": {
                        "value": "light"
                    }
                }
            }
        }
    }
}

def get_page(payload):

    response = requests.post(
        URL,
        headers=headers,
        json=payload
    )

    if response.status_code == 200:
        return response.json()

    else:
        print("Error:", response.status_code)
        return None

In [33]:
def extract_posts(data):

    posts = []

    for item in data["list_widgets"]:

        # فقط آگهی‌ها
        if "data" not in item:
            continue

        post = item["data"]

        posts.append({

            "title": post.get("title"),

            "top_description": 
                post.get("top_description_text"),

            "middle_description":
                post.get("middle_description_text"),

            "bottom_description":
                post.get("bottom_description_text"),

            "token":
                post.get("token"),

            "url":
                f"https://divar.ir/v/{post.get('token')}"

        })


    return posts

In [34]:
def get_all_posts(first_payload, max_pages=10):

    all_posts = []

    payload = first_payload.copy()


    for page in range(max_pages):

        print("Getting page:", page+1)


        data = get_page(payload)


        if data is None:
            break


        posts = extract_posts(data)


        all_posts.extend(posts)


        print(
            "Total posts:",
            len(all_posts)
        )


        if not data["pagination"]["has_next_page"]:
            break


        # آماده کردن صفحه بعد

        payload["pagination_data"] = (
            data["pagination"]["data"]
        )


        time.sleep(1)


    return all_posts

In [35]:
posts = get_all_posts(
    payload,
    max_pages= 5
)

Getting page: 1
Total posts: 24
Getting page: 2
Total posts: 48
Getting page: 3
Total posts: 72
Getting page: 4
Total posts: 96
Getting page: 5
Total posts: 120


In [36]:
df = pd.DataFrame(posts)

print(df.shape)

df.head()

(120, 6)


,title,top_description,middle_description,bottom_description,token,url
0,پژو 206 تیپ ۵,"۱۶۳,۰۰۰ کیلومتر","۹۲۰,۰۰۰,۰۰۰ تومان",در گیشا,ga45a1se,https://divar.ir/v/ga45a1se
1,206 تیپ 5 مدل 89,"۲۹۰,۰۰۰ کیلومتر","۷۰۰,۰۰۰,۰۰۰ تومان",دقایقی پیش در سازمان آب,ga4xLp7G,https://divar.ir/v/ga4xLp7G
2,پژو ۲۰۶ تیپ ۵ بدون رنگ,"۲۸۰,۰۰۰ کیلومتر","۱,۱۰۰,۰۰۰,۰۰۰ تومان",دقایقی پیش در درختی,ga4tbCwN,https://divar.ir/v/ga4tbCwN
3,فروش ۲۰۶ سفید,"۷۵,۰۰۰ کیلومتر","۹۷۰,۰۰۰,۰۰۰ تومان",در سنائی,gagFqX0V,https://divar.ir/v/gagFqX0V
4,پژو ۲۰۶ تیپ ۵ فرانسوی مدل ۹۰,"۲۶۷,۰۰۰ کیلومتر","۸۵۰,۰۰۰,۰۰۰ تومان",در جنت‌آباد شمالی,gaphyEHm,https://divar.ir/v/gaphyEHm


In [37]:
import requests
from bs4 import BeautifulSoup


url = "https://divar.ir/v/%D9%81%D8%B1%D9%88%D8%B4-%DB%B2%DB%B0%DB%B6-%D8%AA%DB%8C%D9%BE-%DB%B5-%D9%85%D8%AF%D9%84-%DB%B9%DB%B8-%D9%81%D9%88%D9%84-%D8%A8%DB%8C-%D8%B1%D9%86%DA%AF-%D8%AF%D8%B1-%D8%AD%D8%AF-%D9%86%D9%88/gaoFcZf_?tracker_session_id=31a1a64b-0035-4b4f-ab9d-5636ec032489_gaoFcZf__N"


headers = {
    "User-Agent": "Mozilla/5.0"
}


response = requests.get(
    url,
    headers=headers
)


soup = BeautifulSoup(
    response.text,
    "html.parser"
)


rows = soup.select(
    ".kt-score-row"
)


for row in rows:
    title = row.select_one(
        ".kt-score-row__title"
    )

    score = row.select_one(
        ".kt-score-row__score"
    )

    if title and score:
        print(
            title.text.strip(),
            ":",
            score.text.strip()
        )

موتور : سالم
وضعیت شاسی‌ها : سالم و پلمپ
بدنه : سالم و بی‌خط و خش
گیربکس : سالم و پلمپ


In [38]:
import requests
from bs4 import BeautifulSoup


def get_car_detail(url):

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(
        url,
        headers=headers
    )

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )


    result = {}


    for row in soup.select(".kt-score-row"):

        title = row.select_one(
            ".kt-score-row__title"
        )

        score = row.select_one(
            ".kt-score-row__score"
        )


        if title and score:

            key = title.text.strip()

            value = score.text.strip()


            result[key] = value


    return result

In [39]:
url = "https://divar.ir/v/%D9%81%D8%B1%D9%88%D8%B4-%DB%B2%DB%B0%DB%B6-%D8%AA%DB%8C%D9%BE-%DB%B5-%D9%85%D8%AF%D9%84-%DB%B9%DB%B8-%D9%81%D9%88%D9%84-%D8%A8%DB%8C-%D8%B1%D9%86%DA%AF-%D8%AF%D8%B1-%D8%AD%D8%AF-%D9%86%D9%88/gaoFcZf_?tracker_session_id=31a1a64b-0035-4b4f-ab9d-5636ec032489_gaoFcZf__N"

detail = get_car_detail(url)

print(detail)

{'موتور': 'سالم', 'وضعیت شاسی\u200cها': 'سالم و پلمپ', 'بدنه': 'سالم و بی\u200cخط و خش', 'گیربکس': 'سالم و پلمپ'}


In [40]:
def merge_post_detail(post, detail):

    return {

        "title": post.get("title"),

        "mileage_raw": 
            post.get("top_description"),

        "price_raw":
            post.get("middle_description"),

        "token":
            post.get("token"),


        "engine":
            detail.get("موتور"),

        "chassis":
            detail.get("وضعیت شاسی‌ها"),

        "body":
            detail.get("بدنه"),

        "gearbox":
            detail.get("گیربکس")
    }

In [41]:
cars = []
for post in posts:
    detail = get_car_detail(post["url"])
    cars.append(merge_post_detail(post, detail))
    time.sleep(1)

car_df = pd.DataFrame(cars)
print(car_df.head())

                          title      mileage_raw            price_raw  \
0                 پژو 206 تیپ ۵  ۱۶۳,۰۰۰ کیلومتر    ۹۲۰,۰۰۰,۰۰۰ تومان   
1              206 تیپ 5 مدل 89  ۲۹۰,۰۰۰ کیلومتر    ۷۰۰,۰۰۰,۰۰۰ تومان   
2        پژو ۲۰۶ تیپ ۵ بدون رنگ  ۲۸۰,۰۰۰ کیلومتر  ۱,۱۰۰,۰۰۰,۰۰۰ تومان   
3                 فروش ۲۰۶ سفید   ۷۵,۰۰۰ کیلومتر    ۹۷۰,۰۰۰,۰۰۰ تومان   
4  پژو ۲۰۶ تیپ ۵ فرانسوی مدل ۹۰  ۲۶۷,۰۰۰ کیلومتر    ۸۵۰,۰۰۰,۰۰۰ تومان   

      token engine chassis  body gearbox  
0  ga45a1se   None    None  None    None  
1  ga4xLp7G   None    None  None    None  
2  ga4tbCwN   None    None  None    None  
3  gagFqX0V   None    None  None    None  
4  gaphyEHm   None    None  None    None  


In [42]:
df.head(20)

,title,top_description,middle_description,bottom_description,token,url
0,پژو 206 تیپ ۵,"۱۶۳,۰۰۰ کیلومتر","۹۲۰,۰۰۰,۰۰۰ تومان",در گیشا,ga45a1se,https://divar.ir/v/ga45a1se
1,206 تیپ 5 مدل 89,"۲۹۰,۰۰۰ کیلومتر","۷۰۰,۰۰۰,۰۰۰ تومان",دقایقی پیش در سازمان آب,ga4xLp7G,https://divar.ir/v/ga4xLp7G
2,پژو ۲۰۶ تیپ ۵ بدون رنگ,"۲۸۰,۰۰۰ کیلومتر","۱,۱۰۰,۰۰۰,۰۰۰ تومان",دقایقی پیش در درختی,ga4tbCwN,https://divar.ir/v/ga4tbCwN
3,فروش ۲۰۶ سفید,"۷۵,۰۰۰ کیلومتر","۹۷۰,۰۰۰,۰۰۰ تومان",در سنائی,gagFqX0V,https://divar.ir/v/gagFqX0V
4,پژو ۲۰۶ تیپ ۵ فرانسوی مدل ۹۰,"۲۶۷,۰۰۰ کیلومتر","۸۵۰,۰۰۰,۰۰۰ تومان",در جنت‌آباد شمالی,gaphyEHm,https://divar.ir/v/gaphyEHm
5,تیپ ۵ مدل ۹۳ بیرنگ شاسی سالم,"۱۸۸,۸۰۰ کیلومتر","۱,۰۲۵,۰۰۰,۰۰۰ تومان",نیم ساعت پیش در دریا,ga4VrEq7,https://divar.ir/v/ga4VrEq7
6,پژو ۲۰۶ مدل ۹۴ تیپ ۵,"۱۴۸,۰۰۰ کیلومتر","۱,۰۶۰,۰۰۰,۰۰۰ تومان",نیم ساعت پیش در آسمان,ga4RbrNI,https://divar.ir/v/ga4RbrNI
7,۲۰۶ تیپ ۵,"۱۶۷,۰۰۰ کیلومتر","۹۷۰,۰۰۰,۰۰۰ تومان",نیم ساعت پیش در سازمان برنامه شمالی,ga4Rr5ci,https://divar.ir/v/ga4Rr5ci
8,یک دستگاه خودرو برای فروش,"۵۰۰,۰۰۰ کیلومتر","۸۵۰,۰۰۰,۰۰۰ تومان",نیم ساعت پیش در فرحزاد,ga4RrDF6,https://divar.ir/v/ga4RrDF6
9,۲۰۶تیپ۵ مدل ۹۶ شاسی سالم‌,"۱۷۵,۰۰۰ کیلومتر","۱,۱۳۰,۰۰۰,۰۰۰ تومان",نیم ساعت پیش در دیلمان,ga4Frc1r,https://divar.ir/v/ga4Frc1r
